# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Sep 15 01:24:36 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671174,35.9,1479534,79.1,1479534,79.1
Vcells,1242566,9.5,8388608,64.0,1978697,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [4]:
semillas <- c(146023, 419921, 453601, 906313, 994481, 100003, 200003, 300007, 400009, 500009)
exp_base <- 5000  

for(i in 1:length(semillas)) {
  
  PARAM <- list()  
  PARAM$semilla_primigenia <- semillas[i]
  PARAM$experimento        <- exp_base + i
  PARAM$dataset            <- "analistajr_competencia_2026.csv.gz"
  
} 

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [27]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# ==============================================================================
# EXPERIMENTO 1: Ratios de Dominio Financiero
# ==============================================================================

# Ratios de Tarjetas de Crédito
if (atributos_presentes(c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"))) {
  dataset[, mconsumo_total_tarjetas := mtarjeta_visa_consumo + mtarjeta_master_consumo]
}

if (atributos_presentes(c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones"))) {
  dataset[, ctrx_total_tarjetas := ctarjeta_visa_transacciones + ctarjeta_master_transacciones]
}

if (atributos_presentes(c("Visa_mpagominimo", "mtarjeta_visa_consumo"))) {
  dataset[, ratio_visa_pagomin_consumo := Visa_mpagominimo / (mtarjeta_visa_consumo + 1)]
}

if (atributos_presentes(c("Master_mpagominimo", "mtarjeta_master_consumo"))) {
  dataset[, ratio_master_pagomin_consumo := Master_mpagominimo / (mtarjeta_master_consumo + 1)]
}

if (atributos_presentes(c("Visa_msaldototal", "Visa_mlimitecompra"))) {
  dataset[, ratio_visa_saldo_limite := Visa_msaldototal / (Visa_mlimitecompra + 1)]
}

if (atributos_presentes(c("Master_msaldototal", "Master_mlimitecompra"))) {
  dataset[, ratio_master_saldo_limite := Master_msaldototal / (Master_mlimitecompra + 1)]
}

# Liquididad y Cuentas
if (atributos_presentes(c("mcaja_ahorro", "mcuenta_corriente"))) {
  dataset[, msaldo_total_liquido := mcaja_ahorro + mcuenta_corriente]
}

if (atributos_presentes(c("mcaja_ahorro", "mcuentas_saldo"))) {
  dataset[, ratio_cajaahorro_cuentas := mcaja_ahorro / (mcuentas_saldo + 1)]
}

if (atributos_presentes(c("mcuenta_corriente", "mcuentas_saldo"))) {
  dataset[, ratio_cuentacorriente_cuentas := mcuenta_corriente / (mcuentas_saldo + 1)]
}

# Interacciones con Payroll e Ingresos
if (atributos_presentes(c("mtarjeta_visa_consumo", "mpayroll"))) {
  dataset[, ratio_consumo_payroll := mtarjeta_visa_consumo / (mpayroll + 1)]
}

if (atributos_presentes(c("mprestamos_personales", "mpayroll"))) {
  dataset[, ratio_prestamos_payroll := mprestamos_personales / (mpayroll + 1)]
}

if (atributos_presentes(c("mrentabilidad", "mpayroll"))) {
  dataset[, mrentabilidad_sobre_payroll := mrentabilidad / (mpayroll + 1)]
}

# Métricas por Producto o Transacción
if (atributos_presentes(c("mcuentas_saldo", "ctrx_quarter"))) {
  dataset[, mtransaccion_promedio_quarter := mcuentas_saldo / (ctrx_quarter + 1)]
}

if (atributos_presentes(c("ctrx_quarter", "cproductos"))) {
  dataset[, ctrx_por_producto := ctrx_quarter / (cproductos + 1)]
}

if (atributos_presentes(c("mrentabilidad", "cproductos"))) {
  dataset[, mrentabilidad_por_producto := mrentabilidad / (cproductos + 1)]
}

# ==============================================================================
# EXPERIMENTO 2: Nuevas variables de dominio (uso, payroll, actividad, liquidez)
# ==============================================================================

# --- Uso y comportamiento de tarjetas ---------------------------------------
if (atributos_presentes(c("mconsumo_total_tarjetas", "ctrx_total_tarjetas"))) {
  dataset[, mconsumo_promedio_trx_tarjeta := mconsumo_total_tarjetas / (ctrx_total_tarjetas + 1)]
}

if (atributos_presentes(c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"))) {
  dataset[, ratio_visa_master_consumo := mtarjeta_visa_consumo / (mtarjeta_master_consumo + 1)]
}

if (atributos_presentes(c("Visa_msaldototal", "Master_msaldototal", "Visa_mlimitecompra", "Master_mlimitecompra"))) {
  dataset[, ratio_uso_limite_total := (Visa_msaldototal + Master_msaldototal) /
            (Visa_mlimitecompra + Master_mlimitecompra + 1)]
}

if (atributos_presentes(c("Visa_Fvencimiento", "foto_mes"))) {
  dataset[, dias_hasta_vto_visa := Visa_Fvencimiento - foto_mes]
}

if (atributos_presentes(c("Master_Fvencimiento", "foto_mes"))) {
  dataset[, dias_hasta_vto_master := Master_Fvencimiento - foto_mes]
}

# --- Ingresos / payroll -------------------------------------------------------
if (atributos_presentes(c("mpayroll_delta1", "mpayroll_delta2"))) {
  dataset[, tendencia_payroll := mpayroll_delta1 - mpayroll_delta2]
}

if (atributos_presentes(c("mpayroll", "mrentabilidad"))) {
  dataset[, ratio_payroll_sobre_rentabilidad := mpayroll / (mrentabilidad + 1)]
}

if (atributos_presentes(c("mpayroll", "mpayroll_lag1"))) {
  dataset[, flag_payroll_caido := as.integer(mpayroll == 0 & mpayroll_lag1 > 0)]
}

# --- Rentabilidad y márgenes ---------------------------------------------------
if (atributos_presentes(c("mactivos_margen", "mpasivos_margen"))) {
  dataset[, ratio_activos_pasivos := mactivos_margen / (mpasivos_margen + 1)]
}

if (atributos_presentes(c("mcomisiones", "mrentabilidad"))) {
  dataset[, ratio_comisiones_sobre_rentabilidad := mcomisiones / (mrentabilidad + 1)]
}

if (atributos_presentes(c("mrentabilidad", "mrentabilidad_lag1", "mrentabilidad_lag2"))) {
  dataset[, volatilidad_rentabilidad := apply(
    cbind(mrentabilidad, mrentabilidad_lag1, mrentabilidad_lag2), 1, sd, na.rm = TRUE)]
}

# --- Antigüedad / ciclo de vida del cliente ------------------------------------
if (atributos_presentes(c("cliente_antiguedad", "cliente_edad"))) {
  dataset[, antiguedad_sobre_edad := cliente_antiguedad / (cliente_edad + 1)]
}

if (atributos_presentes(c("cliente_antiguedad", "cproductos"))) {
  dataset[, antiguedad_por_producto := cliente_antiguedad / (cproductos + 1)]
}

# --- Actividad e inactividad ---------------------------------------------------
if (atributos_presentes(c("ctrx_quarter"))) {
  dataset[, flag_inactivo_trimestre := as.integer(ctrx_quarter == 0)]
}

if (atributos_presentes(c("chomebanking_transacciones", "ctrx_total_tarjetas", "ctrx_quarter"))) {
  dataset[, ratio_homebanking_sobre_actividad := chomebanking_transacciones /
            (ctrx_total_tarjetas + ctrx_quarter + 1)]
}

if (atributos_presentes(c("ctrx_quarter_delta1", "ctrx_quarter_delta2"))) {
  dataset[, tendencia_ctrx := ctrx_quarter_delta1 - ctrx_quarter_delta2]
}

# --- Liquidez y ahorro -----------------------------------------------------------
if (atributos_presentes(c("msaldo_total_liquido", "mpayroll"))) {
  dataset[, ratio_liquido_sobre_ingresos := msaldo_total_liquido / (mpayroll + 1)]
}

if (atributos_presentes(c("mprestamos_personales", "mactivos_margen"))) {
  dataset[, ratio_prestamos_sobre_activos := mprestamos_personales / (mactivos_margen + 1)]
}

# --- Cambios de status (degradación de categoría) --------------------------------
if (atributos_presentes(c("Visa_status_delta1"))) {
  dataset[, flag_downgrade_visa := as.integer(Visa_status_delta1 < 0)]
}

if (atributos_presentes(c("Master_status_delta1"))) {
  dataset[, flag_downgrade_master := as.integer(Master_status_delta1 < 0)]
}

# --- Interacción entre variables top-Gain -----------------------------------------
if (atributos_presentes(c("ctrx_quarter", "mconsumo_total_tarjetas"))) {
  dataset[, interaccion_ctrx_consumo := ctrx_quarter * mconsumo_total_tarjetas]
}

if (atributos_presentes(c("mcaja_ahorro", "mcuenta_corriente"))) {
  dataset[, ratio_ahorro_corriente := mcaja_ahorro / (mcuenta_corriente + 1)]
}

# ==============================================================================
# EXPERIMENTO 3: Interacciones No Lineales y Polinómicas
# Objetivo: Agregar variables bien construidas que capturen relaciones complejas
# ==============================================================================

# Función auxiliar: log que maneja valores negativos
log_con_signo <- function(x) {
  sign(x) * log1p(abs(x))
}

# --- Interacciones multiplicativas entre variables top ---
vars_top <- c("ctrx_quarter", "mpayroll", "mcaja_ahorro",
              "Visa_msaldototal", "cliente_antiguedad")
vars_top <- intersect(vars_top, colnames(dataset))

cat("Generando interacciones no lineales para:", paste(vars_top, collapse=", "), "\n")

for (i in 1:(length(vars_top)-1)) {
  for (j in (i+1):length(vars_top)) {
    v1 <- vars_top[i]
    v2 <- vars_top[j]
    nombre <- paste0("interaccion_", v1, "_x_", v2)
    
    # Usamos log_con_signo en lugar de log1p directo
    dataset[, (nombre) := log_con_signo(get(v1)) * log_con_signo(get(v2))]
  }
}

# --- Variables polinómicas suaves (también corregidas) ---
if ("ctrx_quarter" %in% colnames(dataset))
  dataset[, ctrx_quarter_suave := ctrx_quarter / (abs(ctrx_quarter) + 100)]

if ("mpayroll" %in% colnames(dataset))
  dataset[, mpayroll_log := log_con_signo(mpayroll)]

cat("Variables nuevas agregadas: 12 (10 interacciones + 2 polinómicas)\n")
cat("Transformación aplicada: sign(x) * log1p(abs(x)) para manejar valores negativos\n")

# --- Bloque Adicional: Transaccionalidad y Comportamiento (ctrx_quarter) ----
if (atributos_presentes(c("ctrx_quarter", "cproductos"))) {
  # Intensidad de uso por producto contratado
  dataset[, ctrx_por_producto := ctrx_quarter / (cproductos + 1)]
}

if (atributos_presentes(c("ctrx_quarter", "mcuenta_corriente"))) {
  # Relación transaccional sobre cuenta corriente
    dataset[, ratio_ctrx_cuentacorriente := ctrx_quarter / (log_con_signo(mcuenta_corriente) + 1)]
    dataset[, uso_descubierto_cc := cdescubierto_preacordado / (log_con_signo(mcuenta_corriente) + 1)]
}

if (atributos_presentes(c("ctrx_quarter", "cpayroll_trx"))) {
  # Flag de transaccionalidad alta sin cobro de sueldo
  dataset[, flag_alta_trx_sin_payroll := as.integer(ctrx_quarter > 20 & cpayroll_trx == 0)]
}

if (atributos_presentes(c("ctrx_quarter", "ctrx_quarter_lag2"))) {
  # Tendencia transaccional a 2 meses
  dataset[, ctrx_delta2 := ctrx_quarter - ctrx_quarter_lag2]
}

# --- Bloque Adicional: Liquidez y Ahorro ------------------------------------
if (atributos_presentes(c("mcaja_ahorro", "msaldo_total_liquido"))) {
  # Proporción que representa la caja de ahorro dentro de la liquidez total
  dataset[, prop_cajaahorro_liquido := mcaja_ahorro / (msaldo_total_liquido + 1)]
}

if (atributos_presentes(c("mcuenta_corriente", "mcaja_ahorro"))) {
  # Balance entre cuenta corriente y caja de ahorro
  dataset[, ratio_cc_vs_ca := (mcuenta_corriente + 1) / (mcaja_ahorro + 1)]
}

if (atributos_presentes(c("mplazo_fijo", "msaldo_total_liquido"))) {
  # Preferencia de inversión frente a liquidez inmediata
  dataset[, ratio_plazofijo_liquido := mplazo_fijo / (msaldo_total_liquido + 1)]
}

if (atributos_presentes(c("mcuentas_saldo", "msaldo_total_liquido"))) {
  # Saldo total en cuentas respecto a liquidez
  dataset[, prop_cuentas_liquido := mcuentas_saldo / (msaldo_total_liquido + 1)]
}

# --- Bloque Adicional: Deuda y Consumo de Tarjetas --------------------------
if (atributos_presentes(c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"))) {
  # Mix de consumo entre tarjetas (franquicia dominante)
  dataset[, ratio_visa_master_consumo := (mtarjeta_visa_consumo + 1) / (mtarjeta_master_consumo + 1)]
}

if (atributos_presentes(c("mtarjeta_visa_consumo", "mtarjeta_visa_limite"))) {
  # Uso del límite asignado en Visa
  dataset[, uso_limite_visa := mtarjeta_visa_consumo / (mtarjeta_visa_limite + 1)]
}

if (atributos_presentes(c("mtarjeta_master_consumo", "mtarjeta_master_limite"))) {
  # Uso del límite asignado en Master
  dataset[, uso_limite_master := mtarjeta_master_consumo / (mtarjeta_master_limite + 1)]
}

if (atributos_presentes(c("mconsumo_total_tarjetas", "mpayroll"))) {
  # Nivel de consumo en tarjetas sobre el ingreso por payroll
  dataset[, ratio_consumo_payroll := mconsumo_total_tarjetas / (mpayroll + 1)]
}



# --- Bloque Adicional: Ingresos y Payroll -----------------------------------
if (atributos_presentes(c("mpayroll", "mpayroll_lag1"))) {
  # Variación directa en el ingreso de sueldo respecto al mes anterior
  dataset[, mpayroll_delta1 := mpayroll - mpayroll_lag1]
}

if (atributos_presentes(c("mpayroll", "mprestamos_personales"))) {
  # Relación préstamo personal frente al ingreso por sueldo
  dataset[, ratio_prestamo_payroll := mprestamos_personales / (mpayroll + 1)]
}

if (atributos_presentes(c("cpayroll_trx", "cpayroll_trx_lag1"))) {
  # Caída o aumento en el conteo de acreditaciones de sueldo
  dataset[, cpayroll_trx_delta1 := cpayroll_trx - cpayroll_trx_lag1]
}

# --- Bloque Adicional: Estado e Inestabilidad de Tarjetas -------------------
if (atributos_presentes(c("Visa_status", "Visa_status_lag1"))) {
  # Deterioro o mejora del estado en tarjeta Visa
  dataset[, visa_status_delta := Visa_status - Visa_status_lag1]
}

if (atributos_presentes(c("Master_status", "Master_status_lag1"))) {
  # Deterioro o mejora del estado en tarjeta Master
  dataset[, master_status_delta := Master_status - Master_status_lag1]
}

if (atributos_presentes(c("Visa_financiacion", "mtarjeta_visa_consumo"))) {
  # Proporción de consumo Visa financiado
  dataset[, prop_visa_financiado := Visa_financiacion / (mtarjeta_visa_consumo + 1)]
}

# --- Bloque Adicional: Ratios Combinados y Deltas Lag -----------------------
if (atributos_presentes(c("mcaja_ahorro", "mcaja_ahorro_lag1"))) {
  # Delta relativo de caja de ahorro
  dataset[, mcaja_ahorro_delta_rel := (mcaja_ahorro - mcaja_ahorro_lag1) / (abs(mcaja_ahorro_lag1) + 1)]
}

if (atributos_presentes(c("mconsumo_total_tarjetas", "mconsumo_total_tarjetas_lag1"))) {
  # Delta relativo de consumo en tarjetas
  dataset[, mconsumo_tarjetas_delta_rel := (mconsumo_total_tarjetas - mconsumo_total_tarjetas_lag1) / (abs(mconsumo_total_tarjetas_lag1) + 1)]
}

if (atributos_presentes(c("cdescubierto_preacordado", "cdescubierto_preacordado_lag1"))) {
  # Cambio de comportamiento en uso de descubierto
  dataset[, cdescubierto_delta1 := cdescubierto_preacordado - cdescubierto_preacordado_lag1]
}

if (atributos_presentes(c("mcomisiones", "ctrx_quarter"))) {
  # Costo por transacción realizada
  dataset[, comisiones_por_trx := mcomisiones / (ctrx_quarter + 1)]
}

if (atributos_presentes(c("mdebito_automatico", "msaldo_total_liquido"))) {
  # Compromiso de débitos automáticos sobre la liquidez
  dataset[, ratio_debitoauto_liquido := mdebito_automatico / (msaldo_total_liquido + 1)]
}

if (atributos_presentes(c("cproductos", "cproductos_lag1"))) {
  # Caída directa en la cantidad de productos (fuga parcial)
  dataset[, cproductos_delta1 := cproductos - cproductos_lag1]
}


Generando interacciones no lineales para: ctrx_quarter, mpayroll, mcaja_ahorro, Visa_msaldototal, cliente_antiguedad 
Variables nuevas agregadas: 12 (10 interacciones + 2 polinómicas)
Transformación aplicada: sign(x) * log1p(abs(x)) para manejar valores negativos


In [29]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                                
  [2] "foto_mes"                                         
  [3] "internet"                                         
  [4] "cliente_edad"                                     
  [5] "cliente_antiguedad"                               
  [6] "mrentabilidad"                                    
  [7] "mrentabilidad_annual"                             
  [8] "mcomisiones"                                      
  [9] "mactivos_margen"                                  
 [10] "mpasivos_margen"                                  
 [11] "cproductos"                                       
 [12] "mcuenta_corriente"                                
 [13] "mcaja_ahorro"                                     
 [14] "cdescubierto_preacordado"                         
 [15] "mcuentas_saldo"                                   
 [16] "ctarjeta_visa"                                    
 [17] "ctarjeta_visa_transacciones"                      
 [18] "mtarjeta_visa_consumo"                            
 [19] "ctarjeta_master"                                  
 [20] "ctarjeta_master_transacciones"                    
 [21] "mtarjeta_master_consumo"                          
 [22] "cprestamos_personales"                            
 [23] "mprestamos_personales"                            
 [24] "cpayroll_trx"                                     
 [25] "mpayroll"                                         
 [26] "mttarjeta_visa_debitos_automaticos"               
 [27] "ccomisiones_mantenimiento"                        
 [28] "mcomisiones_mantenimiento"                        
 [29] "ccomisiones_otras"                                
 [30] "mtransferencias_recibidas"                        
 [31] "ccallcenter_transacciones"                        
 [32] "thomebanking"                                     
 [33] "chomebanking_transacciones"                       
 [34] "ctrx_quarter"                                     
 [35] "Master_status"                                    
 [36] "Master_mfinanciacion_limite"                      
 [37] "Master_Fvencimiento"                              
 [38] "Master_msaldototal"                               
 [39] "Master_mlimitecompra"                             
 [40] "Master_fultimo_cierre"                            
 [41] "Master_fechaalta"                                 
 [42] "Master_mconsumototal"                             
 [43] "Master_cconsumos"                                 
 [44] "Master_mpagominimo"                               
 [45] "Visa_status"                                      
 [46] "Visa_mfinanciacion_limite"                        
 [47] "Visa_Fvencimiento"                                
 [48] "Visa_msaldototal"                                 
 [49] "Visa_mlimitecompra"                               
 [50] "Visa_fultimo_cierre"                              
 [51] "Visa_fechaalta"                                   
 [52] "Visa_mconsumototal"                               
 [53] "Visa_cconsumos"                                   
 [54] "Visa_mpagominimo"                                 
 [55] "clase_ternaria"                                   
 [56] "kmes"                                             
 [57] "mpayroll_sobre_edad"                              
 [58] "mconsumo_total_tarjetas"                          
 [59] "ctrx_total_tarjetas"                              
 [60] "ratio_visa_pagomin_consumo"                       
 [61] "ratio_master_pagomin_consumo"                     
 [62] "ratio_visa_saldo_limite"                          
 [63] "ratio_master_saldo_limite"                        
 [64] "msaldo_total_liquido"                             
 [65] "ratio_cajaahorro_cuentas"                         
 [66] "ratio_cuentacorriente_cuentas"                    
 [67] "ratio_consumo_payroll"                            
 [68] "ratio_prestamos_payroll"                          
 [69] "mrentabilidad_sobre_payroll"                      


#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [30]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [31]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [32]:
ncol(dataset)
colnames(dataset)

[1] 528

[1] "numero_de_cliente"                                       
  [2] "foto_mes"                                                
  [3] "internet"                                                
  [4] "cliente_edad"                                            
  [5] "cliente_antiguedad"                                      
  [6] "mrentabilidad"                                           
  [7] "mrentabilidad_annual"                                    
  [8] "mcomisiones"                                             
  [9] "mactivos_margen"                                         
 [10] "mpasivos_margen"                                         
 [11] "cproductos"                                              
 [12] "mcuenta_corriente"                                       
 [13] "mcaja_ahorro"                                            
 [14] "cdescubierto_preacordado"                                
 [15] "mcuentas_saldo"                                          
 [16] "ctarjeta_visa"                                           
 [17] "ctarjeta_visa_transacciones"                             
 [18] "mtarjeta_visa_consumo"                                   
 [19] "ctarjeta_master"                                         
 [20] "ctarjeta_master_transacciones"                           
 [21] "mtarjeta_master_consumo"                                 
 [22] "cprestamos_personales"                                   
 [23] "mprestamos_personales"                                   
 [24] "cpayroll_trx"                                            
 [25] "mpayroll"                                                
 [26] "mttarjeta_visa_debitos_automaticos"                      
 [27] "ccomisiones_mantenimiento"                               
 [28] "mcomisiones_mantenimiento"                               
 [29] "ccomisiones_otras"                                       
 [30] "mtransferencias_recibidas"                               
 [31] "ccallcenter_transacciones"                               
 [32] "thomebanking"                                            
 [33] "chomebanking_transacciones"                              
 [34] "ctrx_quarter"                                            
 [35] "Master_status"                                           
 [36] "Master_mfinanciacion_limite"                             
 [37] "Master_Fvencimiento"                                     
 [38] "Master_msaldototal"                                      
 [39] "Master_mlimitecompra"                                    
 [40] "Master_fultimo_cierre"                                   
 [41] "Master_fechaalta"                                        
 [42] "Master_mconsumototal"                                    
 [43] "Master_cconsumos"                                        
 [44] "Master_mpagominimo"                                      
 [45] "Visa_status"                                             
 [46] "Visa_mfinanciacion_limite"                               
 [47] "Visa_Fvencimiento"                                       
 [48] "Visa_msaldototal"                                        
 [49] "Visa_mlimitecompra"                                      
 [50] "Visa_fultimo_cierre"                                     
 [51] "Visa_fechaalta"                                          
 [52] "Visa_mconsumototal"                                      
 [53] "Visa_cconsumos"                                          
 [54] "Visa_mpagominimo"                                        
 [55] "clase_ternaria"                                          
 [56] "kmes"                                                    
 [57] "mpayroll_sobre_edad"                                     
 [58] "mconsumo_total_tarjetas"                                 
 [59] "ctrx_total_tarjetas"                                     
 [60] "ratio_visa_pagomin_consumo"                              
 [61] "ratio_master_pagomin_consumo"                            
 [62] "ratio_visa_saldo_limite"      

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [33]:
# No se implementa la reduccion de la dimensionalidad con canaritos

#### 9.3.2 Modelado - LOOP DE 10 SEMILLAS

* Hiperparametros 
  * num_leaves = 64
  * min_data_in_leaf = 128
  * feature_fraction = 0.5
  * learning_rate = 0.03

In [34]:
semillas <- c(146023, 419921, 453601,906313,994481, 100003, 200003, 300007, 400009, 500009)

# Contenedor para los resultados de cada semilla
tb_resultado_semillas <- data.table(
  semilla = integer(),
  AUC_grid = numeric(),
  mejores_hiperparametros = list()
)

# ============================================================================
# 9.3.2.1 Training Strategy (COMUN a todas las semillas)
# ============================================================================

PARAM$trainingstrategy$validate  <- c(202107)

PARAM$trainingstrategy$training  <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

PARAM$trainingstrategy$training_pct  <- 1.0
PARAM$trainingstrategy$positivos  <- c("BAJA+1", "BAJA+2")

PARAM$trainingstrategy$final_train  <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)

PARAM$trainingstrategy$future  <- c(202109)

# clase01 (una sola vez, no depende de la semilla)
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0)]

# campos buenos (una sola vez)
campos_buenos  <- copy(setdiff(
  colnames(dataset), c("clase_ternaria", "clase01", "azar", "numero_de_cliente")
))

# fold_final_train (una sola vez)
dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train]

# dfuture (una sola vez)
dfuture  <- dataset[foto_mes %in% PARAM$trainingstrategy$future]

if (!require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

# Kaggle setup (una sola vez)
PARAM$kaggle$competencia  <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes  <- seq(1800, 2400, by = 100)
dir.create("kaggle", showWarnings = FALSE)

# ============================================================================
# LOOP PRINCIPAL: una iteracion por cada semilla
# ============================================================================

for (i_semilla in seq_along(semillas)) {
  
  semilla_actual <- semillas[i_semilla]
  cat("\n\n")
  cat("============================================================\n")
  cat(sprintf("  SEMILLA %d de %d  -->  %d\n", i_semilla, length(semillas), semilla_actual))
  cat("============================================================\n")
  cat(format(Sys.time(), "%a %b %d %X %Y\n"))
  
  # Actualizo la semilla y el experimento en PARAM
  PARAM$semilla_primigenia <- semilla_actual
  PARAM$experimento <- 9300 + i_semilla  # 9101, 9102, 9103, 9104, 9105
  
  # ------------------------------------------------------------------
  # 1) azar + fold_train (depende de la semilla)
  # ------------------------------------------------------------------
  set.seed(semilla_actual, kind = "L'Ecuyer-CMRG")
  dataset[, azar := runif(nrow(dataset))]
  
  dataset[, fold_train := foto_mes %in% PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct)]
  
  # ------------------------------------------------------------------
  # 2) dtrain y dvalidate (se reconstruyen con la nueva semilla)
  # ------------------------------------------------------------------
  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
    label = dataset[fold_train == TRUE, clase01],
    free_raw_data = TRUE
  )
  
  dvalidate <- lgb.Dataset(
    data = data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
    label = dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
    free_raw_data = TRUE
  )
  
  # ------------------------------------------------------------------
  # 3) Parametros fijos de LightGBM (con la semilla actual)
  # ------------------------------------------------------------------
  PARAM$lgbm$param_fijos <- list(
    objective = "binary",
    metric = "auc",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    verbosity = -100,
    force_row_wise = TRUE,
    seed = semilla_actual,
    max_bin = 31,
    learning_rate = 0.03,
    feature_fraction = 0.5,
    num_iterations = 2048,
    early_stopping_rounds = 200,
    num_leaves = 64,
    min_data_in_leaf = 128
  )
  
  # ------------------------------------------------------------------
  # 4) Funcion Estimar_AUC_lightgbm
  # ------------------------------------------------------------------
  Estimar_AUC_lightgbm <- function(x) {
    param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
    
    modelo_train <- lgb.train(
      data = dtrain,
      valids = list(valid = dvalidate),
      eval = "auc",
      param = param_completo,
      verbose = -100
    )
    
    AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]
    niter <- modelo_train$best_iter
    
    rm(modelo_train)
    gc(full = TRUE, verbose = FALSE)
    
    return(list(AUC, niter))
  }
  
  # ------------------------------------------------------------------
  # 5) GRID SEARCH (~65 min por semilla)
  # ------------------------------------------------------------------
  cat("\n--- Grid Search para semilla", semilla_actual, "---\n")
  cat(format(Sys.time(), "Inicio: %a %b %d %X %Y\n"))
  
  tb_nueva <- CJ(
    num_leaves = c(64),
    min_data_in_leaf = c(128),
    feature_fraction = c(0.5)
  )
  
  tb_nueva[, c("AUC", "num_iterations") := Estimar_AUC_lightgbm(.SD),
    by = 1:nrow(tb_nueva)]
  
  fwrite(tb_nueva,
    file = sprintf("tb_grid_search_semilla_%d.txt", semilla_actual),
    sep = "\t")
  
  setorder(tb_nueva, -AUC)
  
  AUC_mejor <- tb_nueva[1, AUC]
  mejores_hp <- as.list(tb_nueva[1])
  mejores_hp$AUC <- NULL
  
  cat(sprintf("\nMejor AUC semilla %d: %.6f\n", semilla_actual, AUC_mejor))
  print(mejores_hp)
  
  # Guardo resultado de esta semilla
  tb_resultado_semillas <- rbind(tb_resultado_semillas,
    data.table(
      semilla = semilla_actual,
      AUC_grid = AUC_mejor,
      mejores_hiperparametros = list(mejores_hp)
    ))
  
  # ------------------------------------------------------------------
  # 6) FINAL TRAINING
  # ------------------------------------------------------------------
  cat("\n--- Final Training para semilla", semilla_actual, "---\n")
  
  dfinal_train <- lgb.Dataset(
    data = data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with = FALSE]),
    label = dataset[fold_final_train == TRUE, clase01],
    free_raw_data = TRUE
  )
  
  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  
  param_final <- c(fijos, mejores_hp)
  
  final_model <- lgb.train(
    data = dfinal_train,
    param = param_final,
    verbose = -100
  )
  
  lgb.save(final_model, sprintf("modelo_semilla_%d.txt", semilla_actual))
  
  tb_importancia <- as.data.table(lgb.importance(final_model))
  fwrite(tb_importancia,
    file = sprintf("importancia_semilla_%d.txt", semilla_actual),
    sep = "\t")
  
  # ------------------------------------------------------------------
  # 7) SCORING sobre future
  # ------------------------------------------------------------------
  prediccion <- predict(
    final_model,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )
  
  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob := prediccion]
  
  fwrite(tb_prediccion,
    file = sprintf("prediccion_semilla_%d.txt", semilla_actual),
    sep = "\t")
  
  # ------------------------------------------------------------------
  # 8) SUBMITS A KAGGLE (7 cortes)
  # ------------------------------------------------------------------
  cat("\n--- Submits a Kaggle para semilla", semilla_actual, "---\n")
  
  setorder(tb_prediccion, -prob)
  
  for (envios in PARAM$kaggle$cortes) {
    tb_prediccion[, Predicted := 0L]
    tb_prediccion[1:envios, Predicted := 1L]
    
    archivo_kaggle <- sprintf("./kaggle/KA%d_%d.csv", PARAM$experimento, envios)
    
    fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
      file = archivo_kaggle,
      sep = ",")
    
    comando <- "kaggle competitions submit "
    competencia <- paste("-c", PARAM$kaggle$competencia)
    arch <- paste("-f", archivo_kaggle)
    mensaje <- sprintf("-m 'envios=%d semilla=%d'", envios, semilla_actual)
    
    linea <- paste(comando, competencia, arch, mensaje)
    salida <- system(linea, intern = TRUE)
    Sys.sleep(30)
    cat(salida, "\n")
  }
  
  # ------------------------------------------------------------------
  # 9) Guardo PARAM.yml de esta semilla
  # ------------------------------------------------------------------
  PARAM$out <- list()
  PARAM$out$lgbm$AUC <- AUC_mejor
  PARAM$out$lgbm$mejores_hiperparametros <- mejores_hp
  
  if (!require("yaml")) install.packages("yaml")
  require("yaml")
  write_yaml(PARAM, file = sprintf("PARAM_semilla_%d.yml", semilla_actual))
  
  # Limpio objetos pesados antes de la siguiente iteracion
  rm(dtrain, dvalidate, dfinal_train, final_model, tb_nueva, tb_prediccion, prediccion)
  gc(full = TRUE, verbose = FALSE)
  
  cat(sprintf("\n*** FIN SEMILLA %d  -  %s ***\n",
    semilla_actual, format(Sys.time(), "%a %b %d %X %Y")))
}

# ============================================================================
# RESUMEN FINAL
# ============================================================================

cat("\n\n")
cat("============================================================\n")
cat("  RESUMEN DE LAS 5 SEMILLAS\n")
cat("============================================================\n")
print(tb_resultado_semillas[, .(semilla, AUC_grid)])

# Guardo tabla resumen final
fwrite(tb_resultado_semillas[, .(semilla, AUC_grid)],
  file = "tb_resumen_5semillas.txt",
  sep = "\t")

cat("\n")
cat(format(Sys.time(), "Fin total: %a %b %d %X %Y\n"))

Loading required package: lightgbm





  SEMILLA 1 de 10  -->  146023
Tue Sep 15 01:24:56 2026

--- Grid Search para semilla 146023 ---
Inicio: Tue Sep 15 01:25:04 2026

Mejor AUC semilla 146023: 0.929753
$num_leaves
[1] 64

$min_data_in_leaf
[1] 128

$feature_fraction
[1] 0.5

$num_iterations
[1] 119


--- Final Training para semilla 146023 ---

--- Submits a Kaggle para semilla 146023 ---
38 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
37 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
36 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
35 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
34 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
33 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
32 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 


Loading required package: yaml




*** FIN SEMILLA 146023  -  Tue Sep 15 01:31:02 2026 ***


  SEMILLA 2 de 10  -->  419921
Tue Sep 15 01:31:02 2026

--- Grid Search para semilla 419921 ---
Inicio: Tue Sep 15 01:31:05 2026

Mejor AUC semilla 419921: 0.931251
$num_leaves
[1] 64

$min_data_in_leaf
[1] 128

$feature_fraction
[1] 0.5

$num_iterations
[1] 211


--- Final Training para semilla 419921 ---

--- Submits a Kaggle para semilla 419921 ---
31 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
30 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
29 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
28 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
27 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
26 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 
25 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

*** FIN SEMILLA 419921  -  Tue S

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit  -c utn-2026-virtual-jr -f ./kaggle/KA9306_2200.csv -m 'envios=2200 semilla=100003'' had status 1”


...

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit  -c utn-2026-virtual-jr -f ./kaggle/KA9310_2100.csv -m 'envios=2100 semilla=500009'' had status 1”


Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit  -c utn-2026-virtual-jr -f ./kaggle/KA9310_2200.csv -m 'envios=2200 semilla=500009'' had status 1”


Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit  -c utn-2026-virtual-jr -f ./kaggle/KA9310_2300.csv -m 'envios=2300 semilla=500009'' had status 1”


Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit  -c utn-2026-virtual-jr -f ./kaggle/KA9310_2400.csv -m 'envios=2400 semilla=500009'' had status 1”


 

*** FIN SEMILLA 500009  -  Tue Sep 15 02:26:36 2026 ***


  RESUMEN DE LAS 5 SEMILLAS
    semilla  AUC_grid
      <num>     <num>
 1:  146023 0.9297528
 2:  419921 0.9312514
 3:  453601 0.9314350
 4:  906313 0.9306433
 5:  994481 0.9313582
 6:  100003 0.9308584
 7:  200003 0.9307588
 8:  300007 0.9310199
 9:  400009 0.9294735
10:  500009 0.9318706

Fin total: Tue Sep 15 02:26:36 2026


In [35]:
# Guardar hiperparámetros ganadores para el notebook 04
PARAM$out <- list()
PARAM$out$lgbm <- list()
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC]
PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_nueva[1])
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL

dir.create("/content/buckets/b1/exp/WF8200", recursive = TRUE, showWarnings = FALSE)
saveRDS(PARAM$out, file = "/content/buckets/b1/exp/WF9903/PARAM_out_post_tuning.rds")
cat("Hiperparámetros guardados en WF8200\n")

ERROR: Error: object 'tb_nueva' not found
